# Data Exploration: SoundCam RIRs and MedleyDB Input Signals

This notebook provides a quick overview of the datasets used in the DDSP-ARE experiments:
- **SoundCam** room impulse responses (moving listener and moving person scenarios)
- **MedleyDB** full-track music mixes used as excitation signals

In [ ]:
import sys
from pathlib import Path

root = Path().resolve().parents[1]  # src/scripts -> src -> repo root
sys.path.insert(0, str(root / 'src'))
sys.path.insert(0, str(root / 'src' / 'external'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio

from utils.common import load_rirs, get_delay_from_ir, discover_input_signals
from utils.plotting import configure_text_rendering, log_smooth_curve

configure_text_rendering()
print(f'Repository root: {root}')

## 1. SoundCam RIRs — Moving Listener Scenario

In [ ]:
rir_dir = root / 'data' / 'SoundCam' / 'moving_listener'
rirs, srs = load_rirs(rir_dir, max_n=8, normalize=False)

print(f'Found {len(rirs)} RIRs in {rir_dir}')
for i, (rir, sr) in enumerate(zip(rirs, srs)):
    delay = get_delay_from_ir(rir, sr)
    print(f'  RIR {i+1}: length={len(rir)} samples, sr={sr} Hz, direct-sound delay={delay} samples ({delay/sr*1000:.1f} ms)')

In [ ]:
# Plot RIR waveforms and magnitude spectra
if rirs:
    sr = srs[0]
    nfft = 2 ** 14
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for i, rir in enumerate(rirs):
        t = np.arange(len(rir)) / sr * 1000
        axes[0].plot(t, rir / (np.max(np.abs(rir)) + 1e-8), alpha=0.7, linewidth=0.8, label=f'RIR {i+1}')

        freqs = np.fft.rfftfreq(nfft, d=1.0/sr)
        H = np.fft.rfft(rir, n=nfft)
        mag_db = 20 * np.log10(np.abs(H) + 1e-8)
        m = freqs > 0
        f_s, mag_s = log_smooth_curve(freqs[m], mag_db[m], window_pts=121)
        axes[1].semilogx(f_s, mag_s, alpha=0.8, linewidth=1.0, label=f'RIR {i+1}')

    axes[0].set_xlabel('Time [ms]')
    axes[0].set_ylabel('Normalised amplitude')
    axes[0].set_title('Room Impulse Responses (time domain)')
    axes[0].set_xlim(0, min(200, rirs[0].shape[-1] / sr * 1000))
    axes[0].legend(fontsize=7)
    axes[0].grid(True, linestyle=':', alpha=0.7)

    axes[1].set_xlim(50, 20000)
    axes[1].set_xlabel('Frequency [Hz]')
    axes[1].set_ylabel('Magnitude [dB]')
    axes[1].set_title('RIR Magnitude Spectra')
    axes[1].legend(fontsize=7)
    axes[1].grid(True, linestyle=':', alpha=0.7)

    fig.tight_layout()
    plt.show()

## 2. SoundCam RIRs — Moving Person Scenario

In [ ]:
rir_dir_mp = root / 'data' / 'SoundCam' / 'moving_person'
rirs_mp, srs_mp = load_rirs(rir_dir_mp, max_n=8, normalize=False)
print(f'Found {len(rirs_mp)} RIRs in {rir_dir_mp}')
for i, (rir, sr) in enumerate(zip(rirs_mp, srs_mp)):
    delay = get_delay_from_ir(rir, sr)
    print(f'  RIR {i+1}: length={len(rir)}, sr={sr}, delay={delay} samples')

## 3. MedleyDB Input Signals

In [ ]:
medleydb_dir = root / 'data' / 'MedleyDB'
audio_files = sorted(medleydb_dir.glob('**/*.wav')) + sorted(medleydb_dir.glob('**/*.mp3'))
print(f'Found {len(audio_files)} audio files in {medleydb_dir}')
for f in audio_files[:10]:
    print(f'  {f.name}')
if len(audio_files) > 10:
    print(f'  ... and {len(audio_files) - 10} more')

In [ ]:
# Load and display statistics for a few songs
if audio_files:
    input_cfg = {'use_songs_folder': True, 'max_num_songs': 4, 'max_audio_len_s': [30.0]}
    signals = discover_input_signals(input_cfg)
    print(f'Discovered {len(signals)} input signals.')
    for mode, info in signals[:4]:
        path = info.get('path', '?')
        print(f'  {mode}: {Path(path).name}')

## 4. Target Response Preview

Preview the Kirkeby compensation target computed from the first RIR.

In [ ]:
from utils.common import get_compensation_EQ_params, build_target_response_lin_phase

if rirs:
    rir = rirs[0]
    sr = srs[0]
    roi = (50.0, 20000.0)

    eq_comp = get_compensation_EQ_params(rir, sr, ROI=roi, num_sections=7)
    target_mag_resp = eq_comp['target_response_db']
    target_mag_freqs = eq_comp['freq_axis_smoothed']
    rir_mag_db_smooth = eq_comp.get('rir_mag_db_smoothed', None)

    fig, ax = plt.subplots(figsize=(8, 4))
    if rir_mag_db_smooth is not None:
        ax.semilogx(target_mag_freqs, rir_mag_db_smooth, 'b--', linewidth=1.2, label='RIR (smoothed)')
    ax.semilogx(target_mag_freqs, target_mag_resp, 'r-', linewidth=1.4, label='Target response')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlim(roi)
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Magnitude [dB]')
    ax.set_title('Compensation target response (from first RIR)')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.7)
    fig.tight_layout()
    plt.show()